In [ ]:
print("hello")

In [ ]:
from langchain import PromptTemplate
from langchain.chains import RetrievalQA
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Pinecone
from langchain.document_loaders import PyMuPDFLoader,DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import PromptTemplate
from langchain.llms import CTransformers
import pinecone


In [ ]:
PINECONE_API_KEY = "pcsk_Lsf9N_BT8EmkszGtiAas1B8mvH8EcJQbaGtZWYFSYuneBhFpnnchYd3L5dvYLnu1tvpaE"

In [ ]:
PINECONE_API_ENV = "gcp-starter"

In [ ]:
from langchain.document_loaders import DirectoryLoader, PyPDFLoader


def load_pdf(data):
    loader = DirectoryLoader(data, glob="*.pdf", loader_cls=PyPDFLoader)  # Use loader_cls
    documents = loader.load()
    return documents


In [ ]:
extracted_data = load_pdf("data/")

In [ ]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=20)
    text_chunks = text_splitter.split_documents(extracted_data)

    return text_chunks

In [ ]:
text_chunks = text_split(extracted_data)

In [ ]:
print("length of the my chunk:",len(text_chunks))

In [ ]:
def download_hugging_face_embedding():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings


In [ ]:
embeddings = download_hugging_face_embedding()

In [ ]:
embeddings

In [ ]:
query_result = embeddings.embed_query("Hello Jyoti")
print("Length",len(query_result))

In [ ]:
text_chunks

In [ ]:
print(type(text_chunks))

In [ ]:
# from langchain.vectorstores import Pinecone as LangChainPinecone
# from langchain.embeddings import HuggingFaceEmbeddings
# from pinecone import Pinecone
# import os

# # Pinecone API Key & Environment
# PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "pcsk_6vt4q7_U3kkYevETBL55ZoogWRtgvPkYeeHHmZP2DPvRbgAbNVnLSEkVpedJXz9shi8CPP") 
# INDEX_NAME = "chatbot"
# PINECONE_ENV = "us-east-1"  # Corrected to match your actual region

# # Initialize Pinecone Client (new SDK)
# pc = Pinecone(api_key=PINECONE_API_KEY)

# # Check if Index Exists Before Connecting
# available_indexes = [index["name"] for index in pc.list_indexes()]
# print("Available indexes:", available_indexes)
# if INDEX_NAME not in available_indexes:
#     raise ValueError(f"Index '{INDEX_NAME}' does not exist. Please create it first.")

# # Get the index
# index = pc.Index(INDEX_NAME)

# # Initialize Embeddings
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# # Create Pinecone vector store using the direct initialization method
# # For the text_key, use the field name that contains your document text
# docsearch = LangChainPinecone(
#     index=index,
#     embedding_function=embeddings.embed_query,
#     text_key="text_chunks"  # Change this if your text field has a different name
# )



In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
from pinecone import Pinecone
import os
import uuid
from typing import List
from langchain.schema import Document

# Pinecone API Key & Environment
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "pcsk_6vt4q7_U3kkYevETBL55ZoogWRtgvPkYeeHHmZP2DPvRbgAbNVnLSEkVpedJXz9shi8CPP") 
INDEX_NAME = "chatbot"

# Initialize Pinecone Client
pc = Pinecone(api_key=PINECONE_API_KEY)

# Connect to the index
index = pc.Index(INDEX_NAME)

# Initialize Embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Function to process documents and add to Pinecone
def add_documents_to_pinecone(documents: List[Document], batch_size: int = 100):
    """
    Process documents, create embeddings, and add to Pinecone
    
    Args:
        documents: List of Document objects with page_content and metadata
        batch_size: Number of documents to process in each batch
    """
    total_docs = len(documents)
    print(f"Processing {total_docs} documents...")
    
    if total_docs == 0:
        print("No documents to process!")
        return
    
    # Process in batches
    for i in range(0, total_docs, batch_size):
        end_idx = min(i + batch_size, total_docs)
        batch = documents[i:end_idx]
        
        # Extract texts for embedding
        texts = [doc.page_content for doc in batch]
        metadatas = [doc.metadata for doc in batch]
        
        # Generate embeddings
        print(f"Generating embeddings for batch {i//batch_size + 1}...")
        vectors = embeddings.embed_documents(texts)
        print(f"Generated {len(vectors)} embeddings for this batch.")
        
        # Prepare data for Pinecone
        pinecone_records = []
        for j, (text, vector, metadata) in enumerate(zip(texts, vectors, metadatas)):
            # Create unique ID
            doc_id = f"doc_{uuid.uuid4()}"
            
            # Add text to metadata
            metadata_with_text = metadata.copy()
            metadata_with_text["text"] = text  # This is the key field for retrieval
            
            # Create record
            record = {
                "id": doc_id,
                "values": vector,
                "metadata": metadata_with_text
            }
            pinecone_records.append(record)
        
        # Upload to Pinecone
        print(f"Uploading batch {i//batch_size + 1} to Pinecone...")
        try:
            index.upsert(vectors=pinecone_records)
            print(f"Successfully uploaded {len(pinecone_records)} records to Pinecone.")
        except Exception as e:
            print(f"Error uploading batch to Pinecone: {e}")
        
        print(f"Processed {end_idx}/{total_docs} documents")
    
    print("All documents processed and uploaded to Pinecone!")

# # Test query function
# def test_search(query_text):
    print(f"\nSearching for: '{query_text}'")
    query_embedding = embeddings.embed_query(query_text)
    
    results = index.query(
        vector=query_embedding,
        top_k=3,
        include_metadata=True
    )
    
    print("\nSearch Results:")
    matches = results.get("matches", [])
    if not matches:
        print("No results found.")
        return
        
    for i, match in enumerate(matches):
        print(f"\nResult {i+1} (Score: {match.get('score', 0):.4f}):")
        metadata = match.get("metadata", {})
        text = metadata.get("text", "No text available")
        print(f"Text: {text[:150]}..." if len(text) > 150 else f"Text: {text}")
        print(f"Source: {metadata.get('source', 'Unknown')}")
        print(f"Page: {metadata.get('page', 'Unknown')}")

# Example usage
if __name__ == "__main__":
    # Assuming `text_chunks` is already defined as a list
    # text_chunks = text_split(extracted_data)
    
    # Verify that text_chunks is a list
    if not isinstance(text_chunks, list):
        raise ValueError("text_chunks must be a list.")
    
    # Convert text_chunks to a list of Document objects
    print("Converting text_chunks to Document objects...")
    documents = []
    for i, text in enumerate(text_chunks):
        if isinstance(text, str):
            # If the item is a string, create a Document object with default metadata
            doc = Document(page_content=text, metadata={"chunk_id": i})
        elif isinstance(text, dict):
            # If the item is a dictionary, assume it has 'page_content' and 'metadata'
            doc = Document(page_content=text.get("page_content", ""), metadata=text.get("metadata", {}))
        else:
            # If the item is neither a string nor a dictionary, skip it
            print(f"Skipping item {i} of type {type(text)}")
            continue
        documents.append(doc)
    
    # Add to Pinecone
    add_documents_to_pinecone(documents)
    
    # # Test a search
    # test_search("What is Allergies")

In [ ]:
# Test query function
def test_search(query_text):
    print(f"\nSearching for: '{query_text}'")
    query_embedding = embeddings.embed_query(query_text)
    
    results = index.query(
        vector=query_embedding,
        top_k=3,
        include_metadata=True
    )
    
    print("\nSearch Results:")
    matches = results.get("matches", [])
    if not matches:
        print("No results found.")
        return
        
    for i, match in enumerate(matches):
        print(f"\nResult {i+1} (Score: {match.get('score', 0):.4f}):")
        metadata = match.get("metadata", {})
        text = metadata.get("text", "No text available")
        print(f"Text: {text[:150]}..." if len(text) > 150 else f"Text: {text}")
        print(f"Source: {metadata.get('source', 'Unknown')}")
        print(f"Page: {metadata.get('page', 'Unknown')}")

# Example usage
if __name__ == "__main__":
    # Test a search
    test_search("What is acnes")

In [ ]:
def retrieve_documents(query_text):
    print(f"\n Searching for: '{query_text}'")
    query_embedding = embeddings.embed_query(query_text)

    results = index.query(
        vector=query_embedding,
        top_k=3,  # Number of results to retrieve
        include_metadata=True
    )

    matches = results.get("matches", [])
    if not matches:
        print("No relevant context found.")
        return ["No relevant context found."]

    retrieved_texts = []
    for match in matches:
        metadata = match.get("metadata", {})
        text = metadata.get("text", "").replace("\n", " ")  # Clean newlines
        retrieved_texts.append(text)

    print("\n Retrieved Contexts:")
    for i, text in enumerate(retrieved_texts):
        preview = text[:150] + "..." if len(text) > 150 else text
        print(f" Context {i+1}: {preview}")

    return retrieved_texts

In [ ]:
prompt_template = """
Use the following information to answer the user's question.
If you don't know the answer, just say that you don't know. Don't make up an answer.

Context:
{context}

----------------------------
Question: {question}

Only return the helpful answer below and nothing else.

Helpful answer:
"""


In [ ]:
PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

In [ ]:
llm=CTransformers(model="model/llama-2-7b-chat.ggmlv3.q4_0.bin",
                 model_type="llama",
                 config={'max_new_tokens':512,
                         'temperature':0.8})

In [ ]:

# ✅ Function to Generate Answer using Llama
def generate_answer(query_text):
    retrieved_texts = retrieve_documents(query_text)

    # Combine retrieved texts into a single context string
    context = "\n".join(retrieved_texts)

    # Format the prompt
    prompt = PROMPT.format(context=context, question=query_text)

    print("\n Formatted Prompt for Llama:\n", prompt)

    # Generate response using Llama
    response = llm(prompt)
    
    print("\n  Chatbot Response:", response)
    return response

In [ ]:
# Example Query Execution
if __name__ == "__main__":
    query = "What is acne?"
    
    # 🔍 Retrieve and generate response
    chatbot_response = generate_answer(query)